# **Preparing Dependancies**

**Nama: Kendrick Filbert**  
**Proyek: Lunar Studio — Generative Image Suite**  
Gunakan Colab: Runtime → Change runtime type → T4 GPU. Jalankan semua sel berurutan.
Notebook mengikuti SD 1.5 sesuai rubrik; SD 3.5 tersedia terpisah di repository portfolio.
Gambar referensi di markdown template bukan hasil eksperimen. Output baru muncul di sel kode.
Seed membantu reproduksibilitas dalam lingkungan yang sama; hasil antar GPU/versi dapat berbeda.


In [ ]:
from pathlib import Path
Path('requirements.txt').write_text("# Python 3.11/3.12. Keep Colab's preinstalled CUDA-enabled torch.\ntorch>=2.4,<3\ndiffusers==0.35.1\ntransformers==4.56.2\naccelerate==1.10.1\nhuggingface-hub==0.34.4\nsafetensors==0.6.2\nnumpy==1.26.4\nPillow==10.4.0\nmatplotlib==3.9.2\npandas==2.2.3\nstreamlit==1.32.2\nstreamlit-drawable-canvas==0.9.3\npyngrok==7.2.12\nsentencepiece==0.2.1\nprotobuf>=3.20,<5\nnbformat==5.10.4\nnbclient==0.10.2\nipykernel>=6.29,<7\nimageio-ffmpeg==0.6.0\n", encoding='utf-8')
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'])
print('Dependensi terpasang. Jika diminta restart oleh Colab, restart lalu lanjut dari sel berikutnya.')

# **Kriteria 1: Melakukan Image Generation dari Teks (Text-to-Image)**

## **Load Base Pipeline Model**

In [ ]:
from pathlib import Path
Path('logic.py').write_text('"""Generative Image Suite — Kendrick Filbert.\n\nSD 1.5 submission backend. Model imports are lazy so geometry tests need no GPU.\nWhite mask pixels are editable; black pixels are preserved exactly.\n"""\nimport gc\nimport threading\nfrom contextlib import contextmanager\nfrom functools import lru_cache\nfrom PIL import Image, ImageDraw, ImageFilter, ImageOps\nimport numpy as np\n\nBASE_MODEL = "stable-diffusion-v1-5/stable-diffusion-v1-5"\nINPAINT_MODEL = "stable-diffusion-v1-5/stable-diffusion-inpainting"\nPROMPT = ("2D digital illustration of a cute astronaut in a white spacesuit, "\n          "standing on the moon, full body on the left, Earth in the starry sky, "\n          "empty lunar ground on the right, clean outlines, colorful digital art")\nNEGATIVE_PROMPT = ("photorealistic, realistic, photograph, 3d render, messy, blurry, "\n                   "low quality, bad art, ugly, sketch, grainy, unfinished, chromatic aberration")\nSCHEDULERS = ("Euler A", "DPM++", "DDIM")\nINFERENCE_LOCK = threading.RLock()\n\n\ndef flush_memory():\n    import torch\n    gc.collect()\n    if torch.cuda.is_available():\n        torch.cuda.empty_cache()\n\n\ndef _configure(pipe):\n    import torch\n    pipe.enable_vae_slicing()\n    pipe.enable_attention_slicing()\n    # Keep weights on CPU between calls; activate only the current pipeline.\n    if not torch.cuda.is_available():\n        pipe.to("cpu")\n    return pipe\n\n\n@lru_cache(maxsize=2)\ndef get_pipeline(task="generation"):\n    import torch\n    from diffusers import StableDiffusionPipeline, StableDiffusionInpaintPipeline\n    if task not in ("generation", "inpainting"):\n        raise ValueError("Task harus generation atau inpainting.")\n    cls = StableDiffusionPipeline if task == "generation" else StableDiffusionInpaintPipeline\n    model = BASE_MODEL if task == "generation" else INPAINT_MODEL\n    dtype = torch.float16 if torch.cuda.is_available() else torch.float32\n    pipe = cls.from_pretrained(model, torch_dtype=dtype, use_safetensors=True)\n    return _configure(pipe)\n\n\ndef load_models_cached():\n    """Compatibility with the template. Prefer lazy get_pipeline in the UI."""\n    return get_pipeline("generation"), get_pipeline("inpainting")\n\n\n@contextmanager\ndef active_pipeline(pipe):\n    import torch\n    with INFERENCE_LOCK:\n        try:\n            if torch.cuda.is_available():\n                pipe.to("cuda")\n            yield pipe\n        finally:\n            if torch.cuda.is_available():\n                pipe.to("cpu")\n            flush_memory()\n\n\ndef load_scheduler(pipe, scheduler_name):\n    from diffusers import (EulerAncestralDiscreteScheduler,\n                           DPMSolverMultistepScheduler, DDIMScheduler)\n    classes = {"Euler A": EulerAncestralDiscreteScheduler,\n               "DPM++": DPMSolverMultistepScheduler, "DDIM": DDIMScheduler}\n    if scheduler_name not in classes:\n        raise ValueError(f"Scheduler tidak dikenal: {scheduler_name}")\n    if not hasattr(pipe, "_original_scheduler_config"):\n        pipe._original_scheduler_config = dict(pipe.scheduler.config)\n    options = {"algorithm_type": "dpmsolver++", "use_karras_sigmas": True} if scheduler_name == "DPM++" else {}\n    pipe.scheduler = classes[scheduler_name].from_config(pipe._original_scheduler_config, **options)\n    return pipe\n\n\nset_scheduler = load_scheduler\n\n\ndef _validate(prompt, seed, steps, cfg, num_images=1):\n    if not isinstance(prompt, str) or not prompt.strip():\n        raise ValueError("Prompt tidak boleh kosong.")\n    if not 0 <= int(seed) <= 2**32 - num_images:\n        raise ValueError("Seed di luar rentang uint32.")\n    if not 1 <= int(steps) <= 100 or not 1 <= int(num_images) <= 4:\n        raise ValueError("Steps 1–100 dan jumlah gambar 1–4.")\n    if not 1 <= float(cfg) <= 20:\n        raise ValueError("CFG harus 1–20.")\n\n\ndef _check_tokens(pipe, prompt):\n    tokens = pipe.tokenizer(prompt, truncation=False).input_ids\n    if len(tokens) > pipe.tokenizer.model_max_length:\n        raise ValueError("Prompt terlalu panjang untuk CLIP. Ringkas menjadi maksimal 75 token isi.")\n\n\ndef generate_image(pipe, prompt, neg_prompt, seed, steps, cfg,\n                   num_images=1, scheduler_name="Euler A"):\n    import torch\n    _validate(prompt, seed, steps, cfg, num_images)\n    _check_tokens(pipe, prompt)\n    _check_tokens(pipe, neg_prompt)\n    # One call, true batch inference; a fresh generator per output.\n    generators = [torch.Generator(device="cpu").manual_seed(int(seed) + i)\n                  for i in range(int(num_images))]\n    with active_pipeline(pipe), torch.inference_mode():\n        load_scheduler(pipe, scheduler_name)\n        output = pipe(prompt=prompt, negative_prompt=neg_prompt,\n                      generator=generators, num_images_per_prompt=int(num_images),\n                      num_inference_steps=int(steps), guidance_scale=float(cfg),\n                      width=512, height=512)\n    if any(getattr(output, "nsfw_content_detected", None) or []):\n        raise ValueError("Safety checker memblokir salah satu hasil; ubah prompt.")\n    return output.images\n\n\ndef generate_simple_image(prompt, negative_prompt=NEGATIVE_PROMPT, seed=222):\n    return generate_image(get_pipeline(), prompt, negative_prompt, seed, 30, 7.5)[0]\n\n\ndef generate_advanced_image(prompt, negative_prompt=NEGATIVE_PROMPT, seed=222,\n                            guidance_scale=9.0, num_inference_step=40):\n    return generate_image(get_pipeline(), prompt, negative_prompt, seed,\n                          num_inference_step, guidance_scale)[0]\n\n\ndef manual_mask(image, box=(285, 230, 485, 450)):\n    """Hardcoded starting rectangle for 512px output; adjust after visual review."""\n    mask = Image.new("L", image.size, 0)\n    ImageDraw.Draw(mask).rectangle(box, fill=255)\n    return mask\n\n\ndef normalize_image(image):\n    image = ImageOps.exif_transpose(image).convert("RGB")\n    w, h = image.size\n    if w < 64 or h < 64 or max(w, h) > 1536:\n        raise ValueError("Ukuran gambar harus 64–1536 piksel per sisi.")\n    if w % 8 or h % 8:\n        raise ValueError("Ukuran gambar harus kelipatan 8; jangan merusak geometri mask.")\n    return image\n\n\ndef run_inpainting(pipe, image, mask, prompt, strength=1.0, seed=9,\n                   steps=40, cfg=9.0, negative_prompt=NEGATIVE_PROMPT,\n                   feather=4):\n    import torch\n    _validate(prompt, seed, steps, cfg)\n    if not 0 < strength <= 1 or int(steps * strength) < 1:\n        raise ValueError("Strength harus menghasilkan minimal satu denoising step.")\n    _check_tokens(pipe, prompt)\n    image = normalize_image(image)\n    mask = mask.convert("L").resize(image.size, Image.Resampling.NEAREST)\n    mask = mask.point(lambda p: 255 if p >= 128 else 0)\n    if mask.getbbox() is None:\n        raise ValueError("Mask kosong: tandai area yang akan diedit.")\n    with active_pipeline(pipe), torch.inference_mode():\n        load_scheduler(pipe, "DPM++")\n        output = pipe(prompt=prompt, negative_prompt=negative_prompt,\n                      image=image, mask_image=mask, width=image.width, height=image.height,\n                      strength=float(strength), guidance_scale=float(cfg),\n                      num_inference_steps=int(steps),\n                      generator=torch.Generator("cpu").manual_seed(int(seed)))\n    if any(getattr(output, "nsfw_content_detected", None) or []):\n        raise ValueError("Safety checker memblokir hasil edit.")\n    # Blur only inside editable pixels, preserving every black-mask pixel.\n    alpha = np.asarray(mask.filter(ImageFilter.GaussianBlur(feather))).copy()\n    alpha[np.asarray(mask) == 0] = 0\n    return Image.composite(output.images[0], image, Image.fromarray(alpha))\n\n\ndef inpaint_engine(image, mask, prompt, **kwargs):\n    return run_inpainting(get_pipeline("inpainting"), image, mask, prompt, **kwargs)\n\n\ndef prepare_outpainting(image, expand_pixels=128, direction="all", overlap=24):\n    """Expand geometrically; overlap lets diffusion blend the original boundary."""\n    image = normalize_image(image)\n    if direction not in ("left", "right", "top", "bottom", "all"):\n        raise ValueError("Arah perluasan tidak valid.")\n    if expand_pixels < 8 or expand_pixels % 8:\n        raise ValueError("Ekspansi harus kelipatan 8 dan minimal 8 piksel.")\n    if not 0 <= overlap < min(image.size) // 2:\n        raise ValueError("Overlap terlalu besar.")\n    left = expand_pixels if direction in ("left", "all") else 0\n    right = expand_pixels if direction in ("right", "all") else 0\n    top = expand_pixels if direction in ("top", "all") else 0\n    bottom = expand_pixels if direction in ("bottom", "all") else 0\n    w, h = image.size\n    if max(w + left + right, h + top + bottom) > 1536:\n        raise ValueError("Kanvas melebihi batas 1536px; gunakan gambar lebih kecil.")\n    # Edge extension avoids a solid-color border interpreted as a picture frame.\n    pixels = np.pad(np.asarray(image), ((top, bottom), (left, right), (0, 0)), mode="edge")\n    canvas = Image.fromarray(pixels).filter(ImageFilter.GaussianBlur(18))\n    canvas.paste(image, (left, top))\n    mask = Image.new("L", canvas.size, 255)\n    keep = (left + (overlap if left else 0), top + (overlap if top else 0),\n            left + w - (overlap if right else 0), top + h - (overlap if bottom else 0))\n    # PIL rectangle includes the final coordinate; subtract one for exclusive bounds.\n    ImageDraw.Draw(mask).rectangle((keep[0], keep[1], keep[2]-1, keep[3]-1), fill=0)\n    return canvas, mask\n\n\n@lru_cache(maxsize=1)\ndef load_segmentation():\n    from transformers import CLIPSegProcessor, CLIPSegForImageSegmentation\n    model_id = "CIDAS/clipseg-rd64-refined"\n    return (CLIPSegProcessor.from_pretrained(model_id),\n            CLIPSegForImageSegmentation.from_pretrained(model_id).eval())\n\n\ndef segmentation_mask(image, text="astronaut", threshold=0.45):\n    import torch\n    if not 0 < threshold < 1:\n        raise ValueError("Threshold harus antara 0 dan 1.")\n    processor, model = load_segmentation()\n    inputs = processor(text=[text], images=[image.convert("RGB")],\n                       padding=True, return_tensors="pt")\n    with torch.inference_mode():\n        logits = model(**inputs).logits.reshape(1, 1, 352, 352)\n        probabilities = torch.nn.functional.interpolate(\n            logits, size=(image.height, image.width), mode="bilinear", align_corners=False\n        ).sigmoid()[0, 0].numpy()\n    mask = Image.fromarray((probabilities >= threshold).astype("uint8") * 255)\n    fraction = np.count_nonzero(np.asarray(mask)) / (image.width * image.height)\n    if not 0.005 <= fraction <= 0.8:\n        raise ValueError(f"Mask segmentasi tidak layak ({fraction:.1%}); sesuaikan teks/threshold.")\n    return mask.filter(ImageFilter.MaxFilter(7)), probabilities\n\n\ndef refine_image(image, prompt, negative_prompt=NEGATIVE_PROMPT, seed=222,\n                 strength=0.25, steps=40, cfg=7.5):\n    import torch\n    from diffusers import StableDiffusionImg2ImgPipeline\n    base = get_pipeline()\n    # Share the base components; never load a second copy of the checkpoint.\n    with INFERENCE_LOCK:\n        refiner = StableDiffusionImg2ImgPipeline(**base.components)\n        _validate(prompt, seed, steps, cfg)\n        if not 0 < strength <= 0.4 or int(steps * strength) < 1:\n            raise ValueError("Refiner strength harus kecil: 0 < strength <= 0.4.")\n        with active_pipeline(refiner), torch.inference_mode():\n            load_scheduler(refiner, "DPM++")\n            output = refiner(prompt=prompt, negative_prompt=negative_prompt,\n                             image=image.convert("RGB"), strength=strength,\n                             num_inference_steps=steps, guidance_scale=cfg,\n                             generator=torch.Generator("cpu").manual_seed(seed))\n        if any(getattr(output, "nsfw_content_detected", None) or []):\n            raise ValueError("Safety checker memblokir hasil refiner.")\n        return output.images[0]\n', encoding='utf-8')
import importlib, logic
importlib.reload(logic)
import torch, numpy as np, matplotlib.pyplot as plt, pandas as pd
import json, time, hashlib, platform, importlib.metadata
from PIL import Image, ImageDraw
from IPython.display import display
if not torch.cuda.is_available():
    raise RuntimeError('Aktifkan T4 GPU: Runtime > Change runtime type, lalu hubungkan kembali.')
torch.backends.cudnn.benchmark = False
torch.backends.cuda.matmul.allow_tf32 = False
ARTIFACTS = Path('artifacts'); ARTIFACTS.mkdir(exist_ok=True)
records = []
PROMPT, NEGATIVE_PROMPT = logic.PROMPT, logic.NEGATIVE_PROMPT
SEED = 222
def show(images, titles, cols=2):
    rows = (len(images)+cols-1)//cols
    fig, axes = plt.subplots(rows, cols, figsize=(6*cols, 6*rows), squeeze=False)
    for ax in axes.flat: ax.axis('off')
    for ax, img, title in zip(axes.flat, images, titles):
        ax.imshow(img, cmap='gray' if img.mode == 'L' else None); ax.set_title(title)
    plt.tight_layout(); plt.show(); plt.close(fig)
def save_result(name, image, **parameters):
    image.save(ARTIFACTS/f'{name}.png')
    entry = dict(name=name, sha256=hashlib.sha256(image.tobytes()).hexdigest(),
                 size=list(image.size), parameters=parameters)
    records.append(entry)
    (ARTIFACTS/'manifest.json').write_text(json.dumps(records, indent=2), encoding='utf-8')
    return image
env = dict(python=platform.python_version(), gpu=torch.cuda.get_device_name(0),
           packages={p:importlib.metadata.version(p) for p in
                     ['torch','diffusers','transformers','Pillow','numpy']})
(ARTIFACTS/'environment.json').write_text(json.dumps(env, indent=2))
print(json.dumps(env, indent=2))
pipe = logic.get_pipeline('generation')
print('Base model:', logic.BASE_MODEL)
print('Prompt tokens:', len(pipe.tokenizer(PROMPT).input_ids)-2)

## **Generate Image (generate_simple_image)**
Pada bagian ini, Anda akan menghasilkan gambar dari teks. Sasarannya adalah menghasilkan gambar yang mendekati gambar contoh di bawah ini, yaitu seorang astronaut di permukaan bulan dengan gaya digital art 2D. Perhatikan empat elemen visual utama berikut:

*   Astronaut berbaju putih
*   Permukaan bulan
*   Bumi di langit
*   Gaya ilustrasi 2D

(bukan foto realistis).

Gunakan Seed bernilai 222 dan negative prompt yang telah ditetapkan. Pastikan prompt yang Anda gunakan sama pada kedua fungsi agar hasilnya dapat dibandingkan secara objektif.

### **Referensi output:**

![image.png](https://assets.cdn.dicoding.com/original/academy/dos-9bcc04fc2ac14deaa53794409ae6bbfb20260728090256.png)

In [ ]:
def generate_simple_image(prompt, negative_prompt, seed):
    return logic.generate_simple_image(prompt, negative_prompt, seed)
simple_image = save_result('01_simple', generate_simple_image(PROMPT, NEGATIVE_PROMPT, SEED),
                          prompt=PROMPT, negative_prompt=NEGATIVE_PROMPT, seed=SEED, steps=30, cfg=7.5)
display(simple_image)

## **Generate Image with Hyperparameter Configuration (generate_advanced_image)**
Tetap gunakan Seed bernilai 222 dan negative prompt yang telah ditetapkan. Pastikan prompt yang Anda gunakan sama pada kedua fungsi agar hasilnya dapat dibandingkan secara objektif.

### **Referensi Output:**

![image.png](https://assets.cdn.dicoding.com/original/academy/dos-36542ab0ea26954556368e12e4b58f2420260728090214.png)

In [ ]:
def generate_advanced_image(prompt, negative_prompt, seed, guidance_scale, num_inference_step):
    return logic.generate_advanced_image(prompt, negative_prompt, seed, guidance_scale, num_inference_step)
advanced_image = save_result('02_advanced', generate_advanced_image(PROMPT, NEGATIVE_PROMPT, SEED, 9.0, 40),
                            prompt=PROMPT, negative_prompt=NEGATIVE_PROMPT, seed=SEED, steps=40, cfg=9.0)
show([simple_image, advanced_image], ['Simple: CFG 7.5 / 30 steps', 'Advanced: CFG 9 / 40 steps'])

## **Guidance Scale Comparison**

In [ ]:
cfg_values = [2.0, 7.5, 16.0]
cfg_images, cfg_seconds = [], []
for cfg in cfg_values:
    start = time.perf_counter()
    img = logic.generate_image(pipe, PROMPT, NEGATIVE_PROMPT, SEED, 30, cfg)[0]
    cfg_seconds.append(round(time.perf_counter()-start, 2))
    cfg_images.append(save_result(f'cfg_{cfg}', img, seed=SEED, steps=30, cfg=cfg, scheduler='Euler A'))
show(cfg_images, [f'CFG {x}' for x in cfg_values], cols=3)
display(pd.DataFrame({'CFG':cfg_values, 'seconds':cfg_seconds}))

### **Guidance Scale Explanation:**

*   **Gambar dengan "Scale" Rendah:**   
*"Jelaskan karakteristik gambar yang dihasilkan, seperti tingkat detail, kesesuaian dengan prompt, dan variasi visual yang terlihat."*

*   **Gambar dengan "Scale" Tinggi:**   
*"Jelaskan perbedaan yang terlihat dibandingkan guidance scale rendah, terutama pada detail gambar dan kedekatannya dengan prompt."*

**Rancangan terkontrol:** prompt, negative prompt, seed 222, Euler A, dan 30 steps tetap; hanya CFG berubah.
**Hipotesis, belum observasi:** CFG rendah dapat melemahkan kepatuhan prompt; CFG terlalu tinggi dapat memperkeras kontras dan memunculkan artefak.
**Observasi aktual:** diisi setelah gambar hasil eksekusi diperiksa, meliputi astronaut, Bumi, detail permukaan bulan, dan artefak. Jangan menyamakan hipotesis dengan hasil.


## **Inference Steps Comparison**

In [ ]:
step_values = [10, 30, 50]
step_images, step_seconds = [], []
for steps in step_values:
    start = time.perf_counter()
    img = logic.generate_image(pipe, PROMPT, NEGATIVE_PROMPT, SEED, steps, 7.5)[0]
    step_seconds.append(round(time.perf_counter()-start, 2))
    step_images.append(save_result(f'steps_{steps}', img, seed=SEED, steps=steps, cfg=7.5, scheduler='Euler A'))
show(step_images, [f'{x} steps' for x in step_values], cols=3)
display(pd.DataFrame({'steps':step_values, 'seconds':step_seconds}))

### **Inference Step Explanation:**

*   **Gambar dengan "Step" Rendah:**  
*"Jelaskan karakteristik gambar yang dihasilkan, seperti tingkat detail, ketajaman, serta kemungkinan munculnya noise atau artefak."*
*   **Gambar dengan "Step" Tinggi:**  
*"Jelaskan perbedaan yang terlihat dibandingkan step rendah, terutama pada detail gambar, kehalusan hasil, dan stabilitas visual."*

**Rancangan terkontrol:** hanya jumlah steps berubah; seed 222, prompt, CFG 7.5, dan Euler A tetap.
**Hipotesis, belum observasi:** lebih banyak steps menambah biaya komputasi dan belum tentu selalu meningkatkan kualitas.
**Observasi aktual:** diisi setelah eksekusi dengan membandingkan tepi astronaut, detail tanah, noise, dan waktu pada tabel.


## **Batch Inference from One Prompt**

In [ ]:
batch_images = logic.generate_image(pipe, PROMPT, NEGATIVE_PROMPT, SEED, 30, 7.5, num_images=4)
assert len(batch_images) == 4
for i, img in enumerate(batch_images):
    save_result(f'batch_{i+1}', img, seed=SEED+i, steps=30, cfg=7.5, batch_size=4)
show(batch_images, [f'Image {i+1} | seed {SEED+i}' for i in range(4)], cols=2)
print('Empat gambar dari satu pemanggilan pipeline dengan num_images_per_prompt=4.')

## **Load Scheduler**

In [ ]:
def load_scheduler(pipe, scheduler_name):
    return logic.load_scheduler(pipe, scheduler_name)
scheduler_images = []
for name in logic.SCHEDULERS:
    img = logic.generate_image(pipe, PROMPT, NEGATIVE_PROMPT, SEED, 30, 7.5, scheduler_name=name)[0]
    scheduler_images.append(save_result('scheduler_'+name.replace('+','p').replace(' ','_'), img,
                                       seed=SEED, steps=30, cfg=7.5, scheduler=name))
show(scheduler_images, list(logic.SCHEDULERS), cols=3)
print('Objek model digunakan ulang; hanya scheduler diganti menggunakan from_config.')

### **Scheduler Comparation:**

*   **Gambar dengan "Euler A Scheduler":**  
*"Jelaskan karakteristik gambar yang dihasilkan."*
*   **Gambar dengan "DPM++ Scheduler":**  
*"Jelaskan karakteristik gambar yang dihasilkan."*
*   **Gambar dengan "DDIM Scheduler":**  
*"Jelaskan karakteristik gambar yang dihasilkan."*

**Rancangan terkontrol:** ketiga scheduler memakai model, prompt, seed 222, CFG 7.5, dan 30 steps yang sama.
DPM++ di sini adalah `DPMSolverMultistepScheduler(algorithm_type="dpmsolver++", use_karras_sigmas=True)`.
**Observasi aktual:** diisi setelah memeriksa perbedaan komposisi, outline, tekstur, dan artefak pada tiga hasil di atas.


# **Kriteria 2: Menyempurnakan Gambar Melalui Image-to-Image**

## **Inpainting**

**Perhatian:** Gunakan gambar hasil `generate_advanced_image()` atau `generate_simple_image()` sebagai `image`, bukan gambar baru.

Hasil Inpainting yang Anda lakukan perlu dibuat semirip mungkin atau setidaknya menambahkan broken satelite seperti pada Gambar yang ditetapkan berikut ini. (Gunakan Seed bernilai 9)

![image.png](https://assets.cdn.dicoding.com/original/academy/dos-3c6ff896b1bf10d4181e63a5ad5ce6ce20260728094120.png)

### **Load Model Inpainting**

In [ ]:
pipe_inpaint = logic.get_pipeline('inpainting')
def inpaint_engine(image, mask, prompt, **kwargs):
    return logic.run_inpainting(pipe_inpaint, image, mask, prompt, **kwargs)
print('Inpainting model khusus:', logic.INPAINT_MODEL)

### **Manual Masking**

In [ ]:
# Koordinat manual eksplisit. Periksa overlay dan ubah jika menutupi astronaut.
# Catat revisi setelah trial-and-error pada gambar hasil aktual.
MANUAL_BOX = (285, 230, 485, 450)
mask_manual = logic.manual_mask(advanced_image, MANUAL_BOX)
overlay = advanced_image.copy(); overlay.paste(Image.new('RGB', overlay.size, 'red'), mask=mask_manual.point(lambda p: p//3))
show([advanced_image, mask_manual, overlay], ['Input advanced', 'Manual mask', 'Overlay area edit'], cols=3)
mask_manual.save(ARTIFACTS/'manual_mask.png')

### **Generate**

In [ ]:
satellite_prompt = '2D digital illustration of a broken satellite with large blue solar panels and bent metal antenna lying on the lunar ground, clean outlines, Earth in starry sky'
manual_result = save_result('03_manual_inpaint', inpaint_engine(advanced_image, mask_manual,
    satellite_prompt, seed=9, strength=1.0, steps=40, cfg=10.0),
    seed=9, strength=1.0, steps=40, cfg=10.0, prompt=satellite_prompt, box=MANUAL_BOX)
show([advanced_image, manual_result], ['Before', 'After: broken satellite'])

## **Inpainting Menggunakan Automasking**

Anda bebas untuk melakukan inpainting di area manapun menggunakan model segmentation. Terapkan inpainting pada gambar hasil inpainting manual di level basic sebelumnya.

### **load Model Segmentation Untuk Masking**

In [ ]:
processor, segmentation_model = logic.load_segmentation()
print('Model segmentasi: CIDAS/clipseg-rd64-refined (CPU)')

### **Masking with Segmentation Model**

In [ ]:
mask_auto, probabilities = logic.segmentation_mask(manual_result, text='astronaut', threshold=0.45)
mask_auto.save(ARTIFACTS/'automatic_mask.png')
show([manual_result, Image.fromarray((probabilities*255).astype('uint8')), mask_auto],
     ['Input hasil inpainting manual', 'Probabilitas CLIPSeg', 'Automatic mask astronaut'], cols=3)
print('Fraksi mask:', round(np.count_nonzero(np.asarray(mask_auto))/np.asarray(mask_auto).size, 4))

### **Generate**

In [ ]:
auto_prompt = '2D illustration of a cute astronaut in a white spacesuit with a gold reflective visor standing on the moon, clean outlines'
auto_result = save_result('04_automatic_inpaint', inpaint_engine(manual_result, mask_auto,
    auto_prompt, seed=9, strength=0.8, steps=40, cfg=8.0), seed=9, prompt=auto_prompt, mask='CLIPSeg astronaut')
show([manual_result, auto_result], ['Before automatic mask edit', 'After automatic mask edit'])

## **Outpainting**

**Perhatian:** Gunakan gambar hasil inpainting (manual atau otomatis) sebagai input pada tahap ini.

Pastikan outpainting yang dilakukan mendekati gambar yang diharapkan di bawah ini dan terlihat natural dan masih berhubungan dengan gambar dasar.

###**Referensi Expected Output:**

![image.png](https://assets.cdn.dicoding.com/original/academy/dos-cd50294c07a68cce26bbd1887553201120260728094308.png)

### **Prepare the Canvas**

In [ ]:
def prepare_outpainting(image, expand_pixels=128, direction='right', overlap=24):
    return logic.prepare_outpainting(image, expand_pixels, direction, overlap)
canvas_side, mask_side = prepare_outpainting(auto_result, 192, 'right')
show([canvas_side, mask_side], ['Canvas: expand right 192px', 'White=edit, black=keep'])

### **Generate**

In [ ]:
side_prompt = '2D digital illustration, American flag on a pole planted on the moon on the right, lunar ground, Earth and starry sky, astronaut and broken satellite on the left, clean outlines'
side_result = save_result('05_outpaint_right', inpaint_engine(canvas_side, mask_side,
    side_prompt, seed=9, strength=1.0, steps=40, cfg=9.0), seed=9, direction='right', expansion=192, prompt=side_prompt)
show([auto_result, side_result], ['Before', 'After: right outpainting'])

## **Outpainting Zoom Out**

**Perhatian:** Gunakan gambar hasil outpainting sebelumnya sebagai input pada tahap ini.

Pada tahap ini Anda tidak diharuskan untuk meniru spesifik satu gambar saja, tetapi berikut beberapa output yang dapat dijadikan referensi.

![Untitled design (2) (1).png](https://assets.cdn.dicoding.com/original/academy/dos-62d99428843b020bd9c4f00f64a21e3f20260728094555.png)

### **Prepare Canvas for Zoom Out**

In [ ]:
canvas_zoom, mask_zoom = prepare_outpainting(side_result, 128, 'all', overlap=32)
show([canvas_zoom, mask_zoom], ['Zoom-out canvas (all directions)', 'Mask with boundary overlap'])
print('Dimensi sebelum/sesudah:', side_result.size, canvas_zoom.size)

### **Generate**

In [ ]:
zoom_prompt = 'wide continuous 2D digital illustration of the lunar surface, craters and rocks in foreground, starry black sky above, Earth in sky, white astronaut beside broken satellite and American flag, clean outlines'
zoom_negative = NEGATIVE_PROMPT + ', frame, border, picture frame, collage'
zoom_result = save_result('06_zoom_out', inpaint_engine(canvas_zoom, mask_zoom, zoom_prompt,
    seed=9, strength=1.0, steps=40, cfg=8.0, negative_prompt=zoom_negative), seed=9, direction='all', expansion=128, prompt=zoom_prompt)
show([side_result, zoom_result], ['Before: one-sided result', 'After: Zoom Out'])
print('Periksa kontinuitas horizon dan tanah. Jika ada bingkai/seam, revisi prompt atau overlap lalu jalankan ulang.')

## **Base + Refiner Image Generation**

In [ ]:
refiner_prompt = '2D digital illustration of a futuristic cyberpunk city at night, neon signs, a traveler in a jacket in foreground, clean detailed artwork'
base_image = logic.generate_image(pipe, refiner_prompt, NEGATIVE_PROMPT, SEED, 30, 7.5, scheduler_name='DPM++')[0]
refined_image = logic.refine_image(base_image, refiner_prompt, seed=SEED, strength=0.25, steps=40)
save_result('07_base', base_image, seed=SEED, prompt=refiner_prompt)
save_result('08_refiner', refined_image, seed=SEED, prompt=refiner_prompt, strength=0.25, steps=40)
show([base_image, refined_image], ['Before (Base)', 'After (Refiner, strength 0.25)'])
print('Refiner menggunakan komponen checkpoint base yang sama, bukan SDXL atau checkpoint tambahan.')
print('Artefak dan parameter tersimpan di:', ARTIFACTS.resolve())
print('Sesudah menulis observasi aktual, File > Download > Download .ipynb agar output ikut tersimpan.')